# Scraping: récupération de la page d'accueil de la RTBF

Dans ce notebook, nous créons un robot qui va ouvrir la page d'accueil du site du journal [La RTBF](https://www.rtbf.be/) et récupérer le titre de tous les articles du jour et les stocker dans un fichier csv.

## Imports

In [ ]:
import os
import re
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd

## Récupération de tous les articles de la page d'accueil



In [ ]:
headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}

articles = []

root_url = f"https://www.rtbf.be"
response = requests.get(root_url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')
for link in soup.find_all("a", {"class":"stretched-link outline-hidden"}):
    title = link.text.strip()
    url = root_url + link.get('href')
    articles.append([url, title])

In [ ]:
# Affichage du nombre d'articles récupérés
len(articles)

In [ ]:
# Affichage des 10 premières entrées
articles[:10]

## Création d'un dataframe avec les liens et les titres des articles


In [ ]:
df = pd.DataFrame(articles, columns=['link', 'title'])
df

In [ ]:
# Sauvegarde du dataframe dans un fichier csv
df.to_csv(f"../data/rtbf_{time.strftime('%Y%m%d')}.csv", index=False)

## Téléchargement d'un article et affichage du texte

In [ ]:
# Récupération du premier lien du dataframe
article_url = df.iloc[0][0]

# Ouverture de la page
response = requests.get(article_url, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')
# Parsing du contenu
title = soup.find("h1").get_text(separator=' ')

paragraphs = []
for paragraph in soup.find_all("p", attrs={"class": None}):
    paragraphs.append(paragraph.get_text(separator=' ').strip())

content = "\n".join(paragraphs)
print(title)
print("==================================")
print(content)

### Nettoyage du texte à l'aide d'expressions régulières

In [ ]:
# Suppression de tout ce qui se trouve entre parenthèses
clean_content = re.sub("\([^\)]+\)", " ", content)
# Suppression des espaces multiples
clean_content = re.sub("\s+", " ", clean_content)

print(content)
print("==================================")
print(clean_content)


### Création d'un fichier avec le contenu de l'article


In [ ]:
with open("../data/rtbf_example.txt", "w") as writer:
    writer.write(f"{title}\n\n{clean_content}")

## Pour en savoir plus

- Le web scraping avec Python: https://realpython.com/beautiful-soup-web-scraper-python/
- Tutoriel sur les expressions régulières: https://www.w3schools.com/python/python_regex.asp

# TP1 — Scraping du corpus CAMille

## Objectif

Ce travail consiste à récupérer automatiquement les 51 fichiers PDF
du corpus CAMille à partir du site web du cours.

Nous utiliserons les bibliothèques Python Requests et BeautifulSoup
pour identifier les liens des documents et automatiser leur téléchargement.

In [1]:
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

In [2]:
# URL du site CAMille
root_url = "https://max.de.wilde.web.ulb.be/camille/"

# Récupération de la page web
response = requests.get(root_url, timeout=30)
response.raise_for_status()

# Analyse du contenu HTML
soup = BeautifulSoup(response.text, "html.parser")

# Liste destinée à conserver les liens PDF
pdf_links = []

# Recherche des liens dans la page
for link in soup.find_all("a", href=True):
    pdf_url = urljoin(root_url, link["href"])

    if pdf_url.lower().split("?")[0].endswith(".pdf"):
        pdf_links.append(pdf_url)

# Suppression des doublons
pdf_links = list(dict.fromkeys(pdf_links))

# Affichage des résultats
print(f"Nombre de PDF trouvés : {len(pdf_links)}")

for pdf_url in pdf_links[:5]:
    print(pdf_url)

Nombre de PDF trouvés : 51
https://max.de.wilde.web.ulb.be/camille/KB_JB230_1892-08-07_01-0003.pdf
https://max.de.wilde.web.ulb.be/camille/KB_JB427_1920-01-10_01-00004.pdf
https://max.de.wilde.web.ulb.be/camille/KB_JB555_1836-02-08_01-00002.pdf
https://max.de.wilde.web.ulb.be/camille/KB_JB638_1860-05-21_01-00002.pdf
https://max.de.wilde.web.ulb.be/camille/KB_JB773_1918-11-30_01-00002.pdf


Expliqu'on ce que fait notre programme

Instruction	             Rôle
requests.get()	         Récupère la page web CAMille.
BeautifulSoup()	         Analyse la structure HTML de la page.
find_all("a", href=True) Recherche les liens hypertextes.
urljoin()	             Construit l'adresse complète de chaque lien.
endswith(".pdf")	     Identifie les liens vers des fichiers PDF.
dict.fromkeys()	         Élimine les doublons éventuels.

In [3]:
from pathlib import Path

print("Répertoire de travail :", Path.cwd())
print("Dossier PDF existant :", (Path.cwd() / "data" / "pdf").exists())

Répertoire de travail : c:\Users\COMPUTER STORES\Desktop\BLOC II\tac\tps\tp1
Dossier PDF existant : False


In [4]:
from pathlib import Path

# Remonter de tp1 vers tps, puis vers tac
project_dir = Path.cwd().parent.parent

# Définir le dossier contenant les PDF
pdf_dir = project_dir / "data" / "pdf"

print("Dossier PDF :", pdf_dir)
print("Dossier existant :", pdf_dir.exists())
print("Nombre de PDF présents :", len(list(pdf_dir.glob("*.pdf"))))

Dossier PDF : c:\Users\COMPUTER STORES\Desktop\BLOC II\tac\data\pdf
Dossier existant : True
Nombre de PDF présents : 51


In [5]:
# Compteurs pour le bilan
downloaded = 0
skipped = 0
failed = 0

# Parcourir les 51 liens PDF récupérés
for pdf_url in pdf_links:

    # Extraire le nom du fichier depuis son URL
    filename = pdf_url.split("/")[-1]

    # Construire le chemin de destination
    destination = pdf_dir / filename

    # Vérifier si le fichier existe déjà
    if destination.exists():
        print(f"Déjà présent : {filename}")
        skipped += 1
        continue

    # Télécharger uniquement les fichiers manquants
    try:
        response = requests.get(pdf_url, timeout=60)
        response.raise_for_status()

        # Enregistrer le PDF
        destination.write_bytes(response.content)

        print(f"Téléchargé : {filename}")
        downloaded += 1

    except requests.RequestException as error:
        print(f"Erreur : {filename} - {error}")
        failed += 1

# Afficher le bilan
print("\n===== BILAN DU SCRAPING =====")
print(f"PDF téléchargés : {downloaded}")
print(f"PDF déjà présents : {skipped}")
print(f"Échecs : {failed}")

Déjà présent : KB_JB230_1892-08-07_01-0003.pdf
Déjà présent : KB_JB427_1920-01-10_01-00004.pdf
Déjà présent : KB_JB555_1836-02-08_01-00002.pdf
Déjà présent : KB_JB638_1860-05-21_01-00002.pdf
Déjà présent : KB_JB773_1918-11-30_01-00002.pdf
Déjà présent : KB_JB838_1887-12-28_01-00003.pdf
Déjà présent : KB_JB230_1903-10-16_01-0002.pdf
Déjà présent : KB_JB427_1933-01-04_01-00003.pdf
Déjà présent : KB_JB555_1899-01-19_01-00003.pdf
Déjà présent : KB_JB638_1902-12-20_01-00002.pdf
Déjà présent : KB_JB773_1933-10-07_01-00007.pdf
Déjà présent : KB_JB838_1911-08-03_01-00006.pdf
Déjà présent : KB_JB230_1913-07-05_01-0001.pdf
Déjà présent : KB_JB427_1949-07-18_01-00008.pdf
Déjà présent : KB_JB555_1940-03-01_01-00004.pdf
Déjà présent : KB_JB638_1946-07-18_01-00003.pdf
Déjà présent : KB_JB773_1950-07-22_01-00010.pdf
Déjà présent : KB_JB838_1943-09-04_01-00002.pdf
Déjà présent : KB_JB258_1884-09-03_01-0003.pdf
Déjà présent : KB_JB449_1846-05-30_01-00002.pdf
Déjà présent : KB_JB567_1857-02-02_01-00003.